# 🔑 Solutions — go or nogo? — the real neuron, step by step

One worked answer per exercise. There is usually more than one correct
way to write these — if your self-check printed ✅, yours is right too.

Read this *after* an attempt: seeing the answer teaches far less than
getting stuck did.

In [ ]:
# Run this first. It loads the lecture neuron's recording —
# from data/ if you downloaded it, otherwise straight from GitHub.
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"
DATA = Path("data")            # same layout as a uv-start project
DATA.mkdir(exist_ok=True)
NEURON_1 = DATA / "neuron_0588_20180828_5_lecture.csv"
NEURON_2 = DATA / "neuron_0406_20180620_8_lecture.csv"
for path in [NEURON_1, NEURON_2]:
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + path.name, path)

# Columns: 0 frame, 1 dff, 2 go, 3 nogo, 4 reward, 5 lick
dff  = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=1)
go   = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=2, dtype=int)
nogo = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=3, dtype=int)

FRAME_RATE = 10.8   # frames per second

print("frame:", np.arange(40, 56))
print("dff:  ", dff[40:56].round(1))
print("go:   ", go[40:56])
print("nogo: ", nogo[40:56])

## Solution — Step 1, 🔬 write it: `n_frames`, `lowest`, `lowest_frame`

In [ ]:
n_frames = dff.size
lowest = dff.min()
lowest_frame = dff.argmin()
print(n_frames, lowest, lowest_frame)

## Solution — Step 1, 🐞 broken: the brackets matter

In [ ]:
peak = dff.max()
print("peak:", round(peak, 1))

## Solution — Step 2, 🔬 write it: `nogo_on_mean`

In [ ]:
nogo_on_mean = round(dff[nogo == 1].mean(), 3)
print("mean while nogo is on:", nogo_on_mean)

## Solution — Step 3, 🐞 broken: the tuple trap

In [ ]:
# DEMO - the frame numbers where go is on
go_on = np.where(go == 1)[0]
print(go_on[:18])
print(len(go_on), "frames with go on")

In [ ]:
nogo_on = np.where(nogo == 1)[0]
print(len(nogo_on), "frames")

## Solution — Step 4, 🔬 write it: `nogo_starts`

In [ ]:
# DEMO - the first frame of each go trial
go_starts = go_on[::9]
print(go_starts[:5])
print(len(go_starts), "go trials")

In [ ]:
nogo_starts = nogo_on[::9]
print(len(nogo_starts), "nogo trials; the first starts at frame", nogo_starts[0])

## Solution — Step 5, 🔬 write it: `second_window`

In [ ]:
start = go_starts[1]
second_window = dff[start - 10 : start + 30]
print(second_window.size, "frames")

## Solution — Step 6, 🐞 broken: the trial at frame 2

In [ ]:
# DEMO - every go window, stacked
go_trials = []
for start in go_starts:
    go_trials.append(dff[start - 10 : start + 30])
go_trials = np.array(go_trials)          # a list of 254 windows -> a (254, 40) grid

print("shape:               ", go_trials.shape)
print("trial 1:             ", go_trials[0].shape)
print("column 10, all trials:", go_trials[:, 10].shape)

In [ ]:
nogo_trials = []
for start in nogo_starts:
    if start >= 10:
        nogo_trials.append(dff[start - 10 : start + 30])
nogo_trials = np.array(nogo_trials)
print(nogo_trials.shape)

## Solution — Step 7, 🔬 write it: `go_response` and `go_peak`

In [ ]:
go_response = go_trials.mean(axis=0)
go_peak = round(go_response[10:].max(), 3)
print(go_response.shape, "-> peak", go_peak)

## Solution — Step 8: the answer

In [ ]:
nogo_response = nogo_trials.mean(axis=0)
nogo_peak = round(nogo_response[10:].max(), 3)
ratio = round(go_peak / nogo_peak, 1)
print("go", go_peak, "| nogo", nogo_peak, "| go is", ratio, "times bigger")

In [ ]:
# Run me - one trial, and the two averages
seconds = np.arange(-10, 30) / FRAME_RATE    # frames from the start, as seconds

plt.figure(figsize=(8, 3.5))
plt.plot(seconds, go_trials[0], color="lightgrey", label="one go trial")
plt.plot(seconds, go_response, lw=2.5, label=f"go — {len(go_trials)} trials averaged")
plt.plot(seconds, nogo_response, lw=2.5, label=f"nogo — {len(nogo_trials)} trials averaged")
plt.axvline(0, color="black", lw=1)                 # the word starts
plt.axvspan(0, 9 / FRAME_RATE, alpha=0.1)           # ...and plays for 9 frames
plt.xlabel("seconds from the start of the word")
plt.ylabel("activity (dF/F)")
plt.legend()
plt.show()